# tealtools playground

Edit the TEAL in **Cell 1**, then re-run **Cell 2** (rebuilds the DB) and **Cell 3** (runs analyses). The DB lives in a temp dir so iterations are cheap; the underlying CodeQL queries get re-evaluated on each rebuild (no cross-iteration cache).

Comment out / uncomment whichever analyses in Cell 3 you care about — the substrate is shared so adding a detector is one line.

In [ ]:
# --- 1. TEAL source ---
# Edit me, then re-run cells 2 and 3.

TEAL_SOURCE = r"""
#pragma version 10
// Sample: external arg flowing into a box write, no validation.

pushbytes "k"
txna ApplicationArgs 0
box_put

pushint 1
return
"""


In [ ]:
# --- 2. Build CodeQL database from the source above ---
import os, shutil, subprocess, tempfile
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".codeql-extractors").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
SEARCH_PATH = REPO_ROOT / ".codeql-extractors"

PLAYGROUND = Path(tempfile.gettempdir()) / "tealtools-playground"
SRC = PLAYGROUND / "src"
DB = PLAYGROUND / "db"

SRC.mkdir(parents=True, exist_ok=True)
(SRC / "prog.teal").write_text(TEAL_SOURCE.lstrip())

if DB.exists():
    shutil.rmtree(DB)

codeql = (
    os.environ.get("CODEQL")
    or shutil.which("codeql")
    or os.path.expanduser("~/tools/codeql/codeql")
)
subprocess.run(
    [codeql, "database", "create", str(DB),
     "--overwrite", "-l", "teal", "-s", str(SRC),
     f"--search-path={SEARCH_PATH}"],
    check=True, capture_output=True,
)
print(f"Built DB at {DB}")


In [ ]:
# --- 3. Run analyses against the freshly-built DB ---
import sys, importlib
sys.path.insert(0, str(REPO_ROOT))

# Force-reload tealtools if you've edited its source between runs
for mod in list(sys.modules):
    if mod == "tealtools" or mod.startswith("tealtools."):
        del sys.modules[mod]

from tealtools import SSAProgram
from tealtools.path_predicates import PathPredicateAnalysis
from tealtools.group_reasoning import analyze as group_shape
from tealtools.cost_analysis import render as render_cost
from tealtools.auth_domination import AuthDominationDetector
from tealtools.nonunique_box_key import NonUniqueBoxKeyDetector
from tealtools.box_dataflow import (
    detect_into_box_flows,
    detect_out_of_box_flows,
    detect_correlated_flows,
)
from tealtools.inner_txn_report import InnerTxnReport

prog = SSAProgram(str(DB))
print(f"SSA: {len(prog.assignments)} assignments, "
      f"{len(prog.blocks)} BBs, {len(prog.phis)} phis\n")

# --- pick the analyses you care about ---

print("=== Path Predicates ===")
print(PathPredicateAnalysis(prog).render())
print()

print("=== Group Shape ===")
print(group_shape(prog).render())
print()

print("=== Cost ===")
print(render_cost(prog))
print()

print("=== Auth Domination ===")
vs = AuthDominationDetector(prog).detect()
print("\n".join(v.pretty() for v in vs) if vs else "(no violations)")
print()

print("=== Box DF: into-box ===")
vs = detect_into_box_flows(prog)
print("\n".join(v.pretty() for v in vs) if vs else "(no violations)")
print()

print("=== Box DF: out-of-box ===")
vs = detect_out_of_box_flows(prog)
print("\n".join(v.pretty() for v in vs) if vs else "(no violations)")
print()

print("=== Inner-txn Report ===")
print(InnerTxnReport(prog).render())
